In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
import numpy as np
import pandas as pd

BASE = "/content/drive/MyDrive/mitacs_runs"

CASES = {
    "Mild": "C_mild",
    "Moderate": "C_moderate",
    "Severe": "C_severe",
}

PPO_MEAN = "PPO (mean over seeds)"
PPO_SD   = "PPO (sd across seeds)"
BASE_ST  = "base-stock (mean)"

routine_rows = []
disruption_rows = []
family_rows = []

for severity, folder in CASES.items():

    p_routine = os.path.join(BASE, folder, "results_comparison.csv")
    p_stress  = os.path.join(BASE, folder, "stress_test_results.csv")

    if not os.path.exists(p_routine):
        print("MISSING:", p_routine)
        continue

    if not os.path.exists(p_stress):
        print("MISSING:", p_stress)
        continue

    # ============================================================
    # 1. ROUTINE CONDITIONS
    # ============================================================
    rc = pd.read_csv(p_routine, index_col=0)

    ppo_J = float(rc.loc["policy_cost_J", PPO_MEAN])
    bs_J  = float(rc.loc["policy_cost_J", BASE_ST])

    routine_rows.append({
        "severity": severity,

        "PPO_cost": round(ppo_J),
        "PPO_sd": round(float(rc.loc["policy_cost_J", PPO_SD])),

        "base_stock_cost": round(bs_J),

        "gap_%": round(100 * (ppo_J - bs_J) / bs_J, 1),

        "PPO_SL": round(float(
            rc.loc["mean_service_level", PPO_MEAN]), 3),

        "base_stock_SL": round(float(
            rc.loc["mean_service_level", BASE_ST]), 3),

        "PPO_expired": round(float(
            rc.loc["expired_units", PPO_MEAN]), 1),

        "base_stock_expired": round(float(
            rc.loc["expired_units", BASE_ST]), 1),
    })

    # ============================================================
    # 2. DISRUPTION CONDITIONS
    # ============================================================
    st = pd.read_csv(p_stress)

    # PPO seeds
    ppo = st[
        st["policy"].astype(str).str.startswith("PPO seed")
    ].copy()

    # Base-stock
    bs = st[
        st["policy"].astype(str).eq("base-stock")
    ].copy()

    # First average every PPO seed
    seed_summary = ppo.groupby("policy").agg(
        J=("policy_cost_J", "mean"),
        SL=("mean_service_level", "mean"),
        expired=("expired_units", "mean"),
        recovery=("recovery", "mean"),
    )

    ppo_disr_J = seed_summary["J"].mean()
    bs_disr_J  = bs["policy_cost_J"].mean()

    disruption_rows.append({
        "severity": severity,

        "PPO_cost": round(ppo_disr_J),
        "PPO_sd": round(seed_summary["J"].std(ddof=1)),

        "base_stock_cost": round(bs_disr_J),

        "gap_%": round(
            100 * (ppo_disr_J - bs_disr_J) / bs_disr_J, 1
        ),

        "PPO_SL": round(seed_summary["SL"].mean(), 3),
        "base_stock_SL": round(
            bs["mean_service_level"].mean(), 3
        ),

        "PPO_expired": round(
            seed_summary["expired"].mean(), 1
        ),

        "base_stock_expired": round(
            bs["expired_units"].mean(), 1
        ),

        "PPO_recovery": round(
            seed_summary["recovery"].mean(), 2
        ),
    })

    # ============================================================
    # 3. RESULTS BY DISRUPTION FAMILY
    # ============================================================
    for family in [
        "line_failure",
        "workforce_shortage",
        "cold_chain"
    ]:

        pf = ppo[ppo["family"] == family]
        bf = bs[bs["family"] == family]

        if len(pf) == 0 or len(bf) == 0:
            continue

        ppo_family = pf.groupby("policy").agg(
            J=("policy_cost_J", "mean"),
            SL=("mean_service_level", "mean"),
            expired=("expired_units", "mean"),
            recovery=("recovery", "mean"),
        )

        pJ = ppo_family["J"].mean()
        bJ = bf["policy_cost_J"].mean()

        family_rows.append({
            "severity": severity,
            "disruption": family,

            "PPO_cost": round(pJ),
            "base_stock_cost": round(bJ),

            "gap_%": round(
                100 * (pJ - bJ) / bJ, 1
            ),

            "PPO_SL": round(
                ppo_family["SL"].mean(), 3
            ),

            "base_stock_SL": round(
                bf["mean_service_level"].mean(), 3
            ),

            "PPO_expired": round(
                ppo_family["expired"].mean(), 1
            ),

            "PPO_recovery": round(
                ppo_family["recovery"].mean(), 2
            ),
        })


# ================================================================
# TABLES
# ================================================================

ROUTINE = pd.DataFrame(routine_rows).set_index("severity")
DISRUPTION = pd.DataFrame(disruption_rows).set_index("severity")
BY_FAMILY = pd.DataFrame(family_rows)

print("\n" + "="*90)
print("ROUTINE CONDITIONS")
print("="*90)
display(ROUTINE)

print("\nPositive gap = PPO is MORE expensive than base-stock.")
print("Negative gap = PPO is CHEAPER than base-stock.")

print("\n" + "="*90)
print("UNDER DISRUPTION")
print("="*90)
display(DISRUPTION)

print("\n" + "="*90)
print("BY DISRUPTION TYPE")
print("="*90)
display(BY_FAMILY)


# ================================================================
# SAVE RESULTS
# ================================================================

ROUTINE.to_csv(
    os.path.join(BASE, "severity_routine_comparison.csv")
)

DISRUPTION.to_csv(
    os.path.join(BASE, "severity_disruption_comparison.csv")
)

BY_FAMILY.to_csv(
    os.path.join(BASE, "severity_by_family.csv"),
    index=False
)

print("\nSaved:")
print("severity_routine_comparison.csv")
print("severity_disruption_comparison.csv")
print("severity_by_family.csv")


ROUTINE CONDITIONS


,PPO_cost,PPO_sd,base_stock_cost,gap_%,PPO_SL,base_stock_SL,PPO_expired,base_stock_expired
severity,,,,,,,,
Mild,29069,10578,26254,10.7,0.806,0.865,24.9,23.1
Moderate,29069,10578,26254,10.7,0.806,0.865,24.9,23.1
Severe,29069,10578,26254,10.7,0.806,0.865,24.9,23.1



Positive gap = PPO is MORE expensive than base-stock.
Negative gap = PPO is CHEAPER than base-stock.

UNDER DISRUPTION


,PPO_cost,PPO_sd,base_stock_cost,gap_%,PPO_SL,base_stock_SL,PPO_expired,base_stock_expired,PPO_recovery
severity,,,,,,,,,
Mild,26402,2290,22777,15.9,0.793,0.858,13.0,31.9,11.95
Moderate,35387,2494,29746,19.0,0.776,0.842,21.4,49.5,11.83
Severe,38962,2312,32006,21.7,0.773,0.838,23.4,44.1,11.91



BY DISRUPTION TYPE


,severity,disruption,PPO_cost,base_stock_cost,gap_%,PPO_SL,base_stock_SL,PPO_expired,PPO_recovery
0,Mild,line_failure,35226,28795,22.3,0.770,0.840,19.9,12.48
1,Mild,workforce_shortage,24041,22060,9.0,0.797,0.861,10.1,NaN
2,Mild,cold_chain,19939,17477,14.1,0.810,0.873,9.0,11.60
3,Moderate,line_failure,43829,34791,26.0,0.757,0.825,36.1,12.08
4,Moderate,workforce_shortage,40022,35106,14.0,0.765,0.830,13.9,11.70
5,Moderate,cold_chain,22310,19341,15.3,0.807,0.870,14.3,11.70
6,Severe,line_failure,43275,35331,22.5,0.755,0.822,29.3,12.43
7,Severe,workforce_shortage,51421,40852,25.9,0.753,0.823,34.3,11.72
8,Severe,cold_chain,22191,19834,11.9,0.809,0.870,6.5,11.59



Saved:
severity_routine_comparison.csv
severity_disruption_comparison.csv
severity_by_family.csv
